<a href="https://colab.research.google.com/github/Raksh1707/taskdeeplearning/blob/main/task10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn


In [2]:
z_dim = 10
img_dim = 28 * 28

# Generator
G = nn.Sequential(
    nn.Linear(z_dim, 64),
    nn.ReLU(),
    nn.Linear(64, img_dim),
    nn.Tanh()
)

# Critic
C = nn.Sequential(
    nn.Linear(img_dim, 64),
    nn.ReLU(),
    nn.Linear(64, 1)
)

In [3]:
opt_G = torch.optim.Adam(G.parameters(), lr=0.001)
opt_C = torch.optim.Adam(C.parameters(), lr=0.001)

In [4]:
def gradient_penalty(real, fake):

    alpha = torch.rand(real.size(0), 1)
    inter = alpha * real + (1 - alpha) * fake
    inter.requires_grad_(True)

    score = C(inter)

    grad = torch.autograd.grad(
        score,
        inter,
        torch.ones_like(score),
        create_graph=True
    )[0]

    return ((grad.norm(2, dim=1) - 1) ** 2).mean()



In [5]:
for epoch in range(1, 101):

    # Real images
    real = torch.randn(32, img_dim)

    # Fake images
    noise = torch.randn(32, z_dim)
    fake = G(noise)

    # Critic loss
    gp = gradient_penalty(real, fake.detach())

    loss_C = C(fake.detach()).mean() - C(real).mean() + 10 * gp

    opt_C.zero_grad()
    loss_C.backward()
    opt_C.step()

    # Generator loss
    fake = G(torch.randn(32, z_dim))

    loss_G = -C(fake).mean()

    opt_G.zero_grad()
    loss_G.backward()
    opt_G.step()

    # Print
    if epoch % 10 == 0:
        print("Epoch:", epoch,
              "Critic Loss:", round(loss_C.item(), 4),
              "Generator Loss:", round(loss_G.item(), 4),
              "Gradient Penalty:", round(gp.item(), 4))

print("WGAN-GP Training Completed!")

Epoch: 10 Critic Loss: 3.6008 Generator Loss: -0.7675 Gradient Penalty: 0.2848
Epoch: 20 Critic Loss: 3.5877 Generator Loss: -3.5772 Gradient Penalty: 0.0269
Epoch: 30 Critic Loss: 5.1043 Generator Loss: -5.2366 Gradient Penalty: 0.0299
Epoch: 40 Critic Loss: 4.7904 Generator Loss: -2.7736 Gradient Penalty: 0.0916
Epoch: 50 Critic Loss: 4.7305 Generator Loss: -0.389 Gradient Penalty: 0.4018
Epoch: 60 Critic Loss: 5.9938 Generator Loss: -0.0896 Gradient Penalty: 0.6167
Epoch: 70 Critic Loss: 6.7657 Generator Loss: -0.0896 Gradient Penalty: 0.7124
Epoch: 80 Critic Loss: 6.4152 Generator Loss: -0.0896 Gradient Penalty: 0.7094
Epoch: 90 Critic Loss: 5.4229 Generator Loss: -0.0896 Gradient Penalty: 0.6581
Epoch: 100 Critic Loss: 5.1931 Generator Loss: -0.0906 Gradient Penalty: 0.6235
WGAN-GP Training Completed!
